# BOT de Ventas — Agente Pandas + Mistral
### Agente inteligente con `create_pandas_dataframe_agent` + contexto de negocio

In [ ]:
# Bloque 1 — Instalaciones
!pip install -q langchain==0.3.25
!pip install -q langchain-core==0.3.63
!pip install -q langchain-community==0.3.24
!pip install -q langchain-mistralai==0.2.10
!pip install -q langchain-experimental==0.3.4
!pip install -q tabulate

print("Todas las dependencias instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 67.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.86 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
# Bloque 2 — Reiniciar el kernel
import os
os.kill(os.getpid(), 9)

In [ ]:
# Bloque 3 — Imports
import os
import pandas as pd
from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents import create_pandas_dataframe_agent

print("Imports correctos")

In [ ]:
# Bloque 4 — API Key y carga del dataset
from google.colab import userdata, files

# API Key desde Secrets de Colab
os.environ['MISTRAL_API_KEY'] = userdata.get('bot-ventas-rag')
print("API Key cargada desde Secrets")

# Subir dataset
uploaded = files.upload()  # Subi tu ventas_normalizado.csv

# Cargar dataset
df = pd.read_csv('ventas_normalizado.csv')
print(f'Dataset cargado: {df.shape[0]} filas | {df.shape[1]} columnas')
df.head(3)

TimeoutException: Requesting secret bot-ventas-rag timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# Bloque 5 — Configurar el LLM Mistral
llm = ChatMistralAI(
    model='mistral-small-latest',
    api_key=os.environ['MISTRAL_API_KEY'],
    temperature=0.3
)

print("Mistral LLM configurado")
print("Modelo: mistral-small-latest")

In [ ]:
# Bloque 6 — Prefix prompt con contexto de negocio
prefix_prompt = """
Eres un asistente experto en analisis de datos de ventas.
Tenes acceso a un DataFrame de pandas llamado `df` con las siguientes columnas:

- ORDERNUMBER: Identificador unico de cada pedido.
- QUANTITYORDERED: Cantidad de unidades pedidas.
- PRICEEACH: Precio de cada unidad del producto.
- SALES: Monto total de la venta (generalmente QUANTITYORDERED * PRICEEACH).
- ORDERDATE: Fecha en la que se realizo el pedido.
- STATUS: Estado actual (ej. Shipped, Cancelled, Resolved, On Hold).
- PRODUCTLINE: Categoria o linea del producto (ej. Motorcycles, Classic Cars).
- CUSTOMERNAME: Nombre del cliente o empresa.
- CITY y COUNTRY: Ubicacion geografica del cliente.

Instrucciones criticas:
1. Para cualquier pregunta numerica o estadistica, DEBES generar y ejecutar codigo Python.
2. Si te preguntan por ventas totales, usa la columna 'SALES'.
3. Si hay valores 'N/A', ignoralos en calculos matematicos para no sesgar los resultados.
4. Responde siempre en espanol de forma clara y concisa.

"""

print("Prefix prompt definido")

In [ ]:
# Bloque 7 — Creacion del agente
agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix_prompt,
    verbose=True,
    allow_dangerous_code=True,
    agent_type='tool-calling'
)

print("Agente configurado con contexto de negocio.")

In [ ]:
# Bloque 8 — Prueba de funcionamiento
pregunta = '¿cuantas filas tiene el dataset?'
respuesta = agent.invoke(pregunta)
print(f"Respuesta del Agente: {respuesta['output']}")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'len(df)'}`


2823El dataset tiene **2823 filas**.

> Finished chain.
Respuesta del Agente: El dataset tiene **2823 filas**.


In [16]:
# Bloque 9 — Interfaz de chat interactiva
import ipywidgets as widgets
from IPython.display import display, HTML

titulo = HTML("<h3 style='color:#1b4080'>BOT de Ventas - Agente Pandas + Mistral</h3><hr>")
chat_output = widgets.Output()
texto_input = widgets.Text(
    placeholder='Escribi tu pregunta aqui...',
    layout=widgets.Layout(width='75%')
)
boton_enviar = widgets.Button(
    description='Enviar',
    button_style='primary',
    layout=widgets.Layout(width='12%')
)
boton_limpiar = widgets.Button(
    description='Limpiar',
    button_style='warning',
    layout=widgets.Layout(width='10%')
)

def mostrar_mensaje(rol, texto):
    if rol == 'usuario':
        color, icono, alineacion = '#2d6a4f', 'Usuario', 'right'
    else:
        color, icono, alineacion = '#1b4080', 'Bot', 'left'
    with chat_output:
        display(HTML(f"""
        <div style='text-align:{alineacion}; margin:8px 0'>
            <span style='background-color:{color}; color:white;
                         padding:10px 15px; border-radius:15px;
                         display:inline-block; max-width:80%;
                         text-align:left; font-size:14px'>
                {icono}: {texto}
            </span>
        </div>
        """))

def enviar(b):
    pregunta = texto_input.value.strip()
    if not pregunta:
        return
    texto_input.value = ''
    mostrar_mensaje('usuario', pregunta)
    try:
        resultado = agent.invoke(pregunta)
        respuesta = resultado['output']
    except Exception as e:
        respuesta = f'Error: {str(e)}'
    mostrar_mensaje('bot', respuesta)

def limpiar(b):
    chat_output.clear_output()

boton_enviar.on_click(enviar)
boton_limpiar.on_click(limpiar)
texto_input.on_submit(enviar)

fila_botones = widgets.HBox([texto_input, boton_enviar, boton_limpiar])
display(titulo, chat_output, fila_botones)

Output()



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "top_paises = df['COUNTRY'].value_counts().head(10)\ntop_paises"}`


COUNTRY
USA          1004
SPAIN         342
FRANCE        314
AUSTRALIA     185
UK            144
ITALY         113
FINLAND        92
NORWAY         85
SINGAPORE      79
CANADA         70
Name: count, dtype: int64Los **10 países que más compran** en el dataset son:

1. **USA** - 1004 pedidos
2. **SPAIN** - 342 pedidos
3. **FRANCE** - 314 pedidos
4. **AUSTRALIA** - 185 pedidos
5. **UK** - 144 pedidos
6. **ITALY** - 113 pedidos
7. **FINLAND** - 92 pedidos
8. **NORWAY** - 85 pedidos
9. **SINGAPORE** - 79 pedidos
10. **CANADA** - 70 pedidos

> Finished chain.


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "productos_mas_vendidos = df.groupby('PRODUCTLINE')['QUANTITYORDERED'].sum().sort_values(ascending=False)\nproductos_mas_vendidos"}`


PRODUCTLINE
CLASSIC CARS        33992
VINTAGE CARS        21069
